# Analysis

Reproduces every statistic, table, and in-text number in the main paper and the
Supplementary Material from the run-level data in `data/`. Each reported value is
checked against the number printed in the manuscript; the final cell summarises
the checks. Run all cells with this folder as the working directory. Tables are
written to `results/`.

| Section | Reproduces |
|---|---|
| 1 | Table 3 (success by factor level), Table 4 (failures), in-text counts |
| 2 | Primary Firth model: fitted probabilities, Table 5, replicate check, model-order fits |
| 3 | Condition-level results: Table 6, Supplementary Table S5 |
| 4 | Run-order check (Supplementary Section S2) |
| 5 | Transformed-count ANOVA, reduced model, Tukey (Supplementary Section S3, Tables S6–S8) |
| 6 | Conditional prediction checks (Supplementary Section S4, Table S9) |
| 7 | Optical comparison (Supplementary Section S5, Table S10) |

Inputs, all taken from the original workbook `raw/Experimentation1.xlsm`:

- `data/attempt_level_outcomes.csv`: the 1280 attempts from sheet `Final`, one row per
  attempt. Failed attempts are colour-coded by category in the workbook; the category is
  written out here.
- `data/run_level_counts.csv`: the same attempts summed to the 128 ten-attempt runs.
- `data/instrument_specific_power_ratios.csv`: sheet `Reflectivity`.

In [1]:
from pathlib import Path
import itertools, platform
import numpy as np
import pandas as pd
import scipy, statsmodels
import statsmodels.api as sm
from scipy.special import expit, logit
from scipy.optimize import minimize
from scipy.stats import beta, chi2, f as fdist, studentized_range
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

ROOT = Path.cwd()
DATA = ROOT / "data"
RES = ROOT / "results"
RES.mkdir(exist_ok=True)
pd.set_option("display.width", 140)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| statsmodels", statsmodels.__version__)

try:
    display
except NameError:
    display = print

CHECKS = []
def check(label, value, reported, tol):
    """Record whether a computed value matches the number printed in the manuscript."""
    ok = abs(float(value) - float(reported)) <= tol
    CHECKS.append(dict(item=label, computed=float(value), reported=float(reported), ok=ok))
    if not ok:
        print(f"MISMATCH  {label}: computed {value!r}, reported {reported!r}")
    return ok

runs = pd.read_csv(DATA / "run_level_counts.csv")
FAIL = ["Recog", "Orient", "Mate", "Focus"]
KEYS = ["Color", "Config", "Ambient", "Height"]
assert len(runs) == 128 and runs.Sl.nunique() == 128
assert (runs[["Success"] + FAIL].sum(axis=1) == 10).all()
assert runs.groupby(KEYS).size().eq(2).all() and not runs.duplicated(KEYS + ["Rep"]).any()
COLORS = ["Lime Green", "Pink", "Red", "Royal Blue"]
CONFIGS = ["Cylindrical", "Hexagonal", "Rectangular", "Trapezium"]
print("128 runs, 64 conditions x 2 replicate runs, every run sums to 10 attempts.")

attempts = pd.read_csv(DATA / "attempt_level_outcomes.csv")
CATEGORY = {"bottom_part_not_recognized": "Recog", "wrong_orientation": "Orient",
            "mating_failed_correct_orientation": "Mate", "could_not_focus": "Focus"}
assert len(attempts) == 1280 and attempts.groupby("Sl").Attempt.nunique().eq(10).all()
assert (attempts.Success == 1).eq(attempts.Failure_category.isna()).all()
summed = pd.crosstab(attempts.Sl, attempts.Failure_category.map(CATEGORY)).reindex(index=runs.Sl, columns=FAIL, fill_value=0)
summed["Success"] = attempts.groupby("Sl").Success.sum()
assert summed.loc[runs.Sl, ["Success"] + FAIL].to_numpy().tolist() == runs[["Success"] + FAIL].to_numpy().tolist()
print("Attempt-level records sum exactly to the run-level counts.")

Python 3.13.17 | numpy 2.5.3 | pandas 3.0.5 | scipy 1.18.1 | statsmodels 0.15.0
128 runs, 64 conditions x 2 replicate runs, every run sums to 10 attempts.
Attempt-level records sum exactly to the run-level counts.


## 1. Assembly outcomes and failures (Section 4.1; Tables 3 and 4)

In [2]:
total = runs.Success.sum()
check("Completed assemblies (of 1280)", total, 1054, 0)
check("Overall success rate (%)", 100 * total / 1280, 82.34, 0.005)

rows = []
for factor in KEYS:
    g = runs.groupby(factor).Success.agg(["sum", "size"])
    for level, r in g.iterrows():
        rows.append(dict(Factor=factor, Level=level, Attempts=10 * r["size"],
                         Successes=r["sum"], Rate_pct=100 * r["sum"] / (10 * r["size"])))
table3 = pd.DataFrame(rows)
table3.to_csv(RES / "Table3_success_by_level.csv", index=False)
display(table3.round(2))
t3 = table3.set_index(["Factor", "Level"]).Rate_pct
for (fac, lev), rep in {("Color", "Lime Green"): 76.56, ("Color", "Pink"): 76.56, ("Color", "Red"): 94.06,
                        ("Color", "Royal Blue"): 82.19, ("Config", "Cylindrical"): 62.81,
                        ("Config", "Hexagonal"): 78.75, ("Config", "Rectangular"): 90.00,
                        ("Config", "Trapezium"): 97.81, ("Ambient", 110): 85.47, ("Ambient", 360): 79.22,
                        ("Height", 250): 88.75, ("Height", 300): 75.94}.items():
    check(f"Table 3 rate {fac}={lev} (%)", t3[(fac, lev)], rep, 0.005)

table4 = pd.concat({f: runs.groupby(f)[FAIL].sum() for f in KEYS})
table4["Total"] = table4.sum(axis=1)
table4.to_csv(RES / "Table4_failures_by_level.csv")
display(table4)
tot = runs[FAIL].sum()
for k, rep in {"Recog": 28, "Orient": 178, "Mate": 11, "Focus": 9}.items():
    check(f"Total {k} failures", tot[k], rep, 0)
check("Total failures", tot.sum(), 226, 0)
check("Wrong orientation share of failures (%)", 100 * tot.Orient / tot.sum(), 78.76, 0.005)
cfg = runs.groupby("Config")[FAIL].sum()
check("Orientation failures, cylindrical", cfg.loc["Cylindrical", "Orient"], 104, 0)
check("Orientation failures, hexagonal", cfg.loc["Hexagonal", "Orient"], 49, 0)
check("All failures, trapezium", cfg.loc["Trapezium"].sum(), 7, 0)
amb = runs.groupby("Ambient")[FAIL].sum(); hgt = runs.groupby("Height")[FAIL].sum()
check("Extra failures at 360 lx", amb.loc[360].sum() - amb.loc[110].sum(), 40, 0)
check("Bottom-part recognition failures at 360 lx", amb.loc[360, "Recog"], 26, 0)
check("Bottom-part recognition failures at 110 lx", amb.loc[110, "Recog"], 2, 0)
check("Extra failures at 300 mm", hgt.loc[300].sum() - hgt.loc[250].sum(), 82, 0)
check("Orientation failures at 300 mm", hgt.loc[300, "Orient"], 123, 0)
check("Orientation failures at 250 mm", hgt.loc[250, "Orient"], 55, 0)

# Counts quoted in the Abstract, Discussion and Conclusion
cc = runs.pivot_table(index="Color", columns="Config", values="Success", aggfunc="sum")
display(cc)
check("Red, minimum successes over configurations (of 80)", cc.loc["Red"].min(), 73, 0)
others = cc.drop(index="Red")
check("Non-red colors on keyed parts, minimum (of 80)", others[["Cylindrical", "Hexagonal"]].min().min(), 34, 0)
check("Non-red colors on keyed parts, maximum (of 80)", others[["Cylindrical", "Hexagonal"]].max().max(), 62, 0)
check("Non-red colors on unkeyed parts, minimum (of 80)", others[["Rectangular", "Trapezium"]].min().min(), 70, 0)
check("Non-red colors on unkeyed parts, maximum (of 80)", others[["Rectangular", "Trapezium"]].max().max(), 80, 0)
nonred = runs[runs.Color != "Red"]
check("Non-red failures", nonred[FAIL].sum().sum(), 207, 0)
check("Non-red orientation failures", nonred.Orient.sum(), 169, 0)
ch = runs.pivot_table(index="Config", columns="Height", values="Success", aggfunc="sum")
check("Cylindrical successes at 250 mm (of 160)", ch.loc["Cylindrical", 250], 129, 0)
check("Cylindrical successes at 300 mm (of 160)", ch.loc["Cylindrical", 300], 72, 0)
check("Trapezium successes at 250 mm (of 160)", ch.loc["Trapezium", 250], 156, 0)
check("Trapezium successes at 300 mm (of 160)", ch.loc["Trapezium", 300], 157, 0)
check("Cylindrical drop 250 to 300 mm (points)",
      100 * (ch.loc["Cylindrical", 250] - ch.loc["Cylindrical", 300]) / 160, 35.6, 0.05)

,Factor,Level,Attempts,Successes,Rate_pct
0,Color,Lime Green,320,245,76.56
1,Color,Pink,320,245,76.56
2,Color,Red,320,301,94.06
3,Color,Royal Blue,320,263,82.19
4,Config,Cylindrical,320,201,62.81
5,Config,Hexagonal,320,252,78.75
6,Config,Rectangular,320,288,90.00
7,Config,Trapezium,320,313,97.81
8,Ambient,110,640,547,85.47
9,Ambient,360,640,507,79.22


Recog  Orient  Mate  Focus  Total
Color   Lime Green      14      55     3      3     75
        Pink             9      61     4      1     75
        Red              3       9     4      3     19
        Royal Blue       2      53     0      2     57
Config  Cylindrical      8     104     5      2    119
        Hexagonal       14      49     0      5     68
        Rectangular      4      23     5      0     32
        Trapezium        2       2     1      2      7
Ambient 110              2      85     5      1     93
        360             26      93     6      8    133
Height  250              9      55     3      5     72
        300             19     123     8      4    154

Config,Cylindrical,Hexagonal,Rectangular,Trapezium
Color,,,,
Lime Green,41,54,72,78
Pink,34,62,73,76
Red,75,74,73,79
Royal Blue,51,62,70,80


True

## 2. Primary analysis: full-factorial Firth binomial model (Section 4.2; Table 5)

Both runs of a condition share one success probability, so the model is fitted to the
condition totals $s_c$ out of 20. With a separate predictor for each condition, the
Firth objective equals a binomial log-likelihood for $s_c+0.5$ successes out of 21.
Each term is tested by setting its sum-coded coefficients to zero while keeping all
other terms and the full-model penalty.

In [3]:
cond = (runs.groupby(KEYS, sort=True)
        .agg(s=("Success", "sum"), pair=("Success", list)).reset_index())
y = cond.s.to_numpy(float); n = np.full(64, 20.0)
LEVELS = [COLORS, CONFIGS, [110, 360], [250, 300]]
def sum_code(series, levels):
    codes = np.vstack([np.eye(len(levels) - 1), -np.ones(len(levels) - 1)])
    return np.array([codes[levels.index(v)] for v in series])
codes = [sum_code(cond[k], lv) for k, lv in zip(KEYS, LEVELS)]
NAMES = ["C", "S", "A", "H"]
terms = {}
for order in range(1, 5):
    for ids in itertools.combinations(range(4), order):
        z = np.ones((64, 1))
        for i in ids:
            z = np.einsum("ij,ik->ijk", z, codes[i]).reshape(64, -1)
        terms[":".join(NAMES[i] for i in ids)] = z
X = np.column_stack([np.ones(64), *terms.values()])
assert np.linalg.matrix_rank(X) == 64
slices, start = {}, 1
for t, z in terms.items():
    slices[t] = np.arange(start, start + z.shape[1]); start += z.shape[1]

p_firth = (y + 0.5) / (n + 1)
b_full = np.linalg.solve(X, logit(p_firth))
def objective(b, Z):  # negative augmented log-likelihood
    eta = Z @ b
    return np.sum((n + 1) * np.logaddexp(0, eta) - (y + 0.5) * eta)
full_obj = objective(b_full, X)
check("Fitted probability at 20/20", p_firth[y == 20][0], 0.9762, 0.00005)

rows = []
for t, idx in slices.items():
    Z = X[:, np.setdiff1d(np.arange(64), idx)]
    fit = sm.GLM(np.column_stack([y + 0.5, n - y + 0.5]), Z,
                 family=sm.families.Binomial()).fit(maxiter=200, tol=1e-12)
    alt = minimize(objective, np.zeros(Z.shape[1]), args=(Z,), method="BFGS",
                   jac=lambda b, Z=Z: Z.T @ ((n + 1) * expit(Z @ b) - (y + 0.5)),
                   options={"gtol": 1e-8, "maxiter": 2000})
    assert np.max(np.abs(expit(Z @ fit.params) - expit(Z @ alt.x))) < 1e-6  # independent optimizer agrees
    D = max(0.0, 2 * (objective(fit.params, Z) - full_obj))
    rows.append(dict(Term=t.replace(":", "x"), DF=len(idx), D=D, p=chi2.sf(D, len(idx))))
table5 = pd.DataFrame(rows)
table5["p_BH"] = multipletests(table5.p, method="fdr_bh")[1]
table5.to_csv(RES / "Table5_Firth_term_tests.csv", index=False)
display(table5.round(4))
REPORTED_T5 = {"C": (13.73, 0.012), "S": (45.31, 0.0), "A": (4.20, 0.121), "H": (11.36, 0.006),
               "CxS": (25.88, 0.011), "CxA": (3.56, 0.391), "CxH": (3.96, 0.391), "SxA": (7.01, 0.160),
               "SxH": (6.91, 0.160), "AxH": (0.26, 0.706), "CxSxA": (10.68, 0.391), "CxSxH": (11.14, 0.391),
               "CxAxH": (0.29, 0.961), "SxAxH": (6.50, 0.168), "CxSxAxH": (6.57, 0.730)}
t5 = table5.set_index("Term")
for term, (D_rep, pbh_rep) in REPORTED_T5.items():
    check(f"Table 5 D, {term}", t5.loc[term, "D"], D_rep, 0.005)
    check(f"Table 5 p_BH, {term}", t5.loc[term, "p_BH"], pbh_rep, 0.0005 if pbh_rep > 0 else 0.0005)
check("Illuminance unadjusted p", t5.loc["A", "p"], 0.040, 0.0005)
check("Configuration x height unadjusted p", t5.loc["SxH", "p"], 0.075, 0.0005)

,Term,DF,D,p,p_BH
0,C,3,13.7256,0.0033,0.0124
1,S,3,45.3143,0.0000,0.0000
2,A,1,4.2002,0.0404,0.1213
3,H,1,11.3585,0.0008,0.0056
4,CxS,9,25.8773,0.0021,0.0107
5,CxA,3,3.5600,0.3131,0.3913
6,CxH,3,3.9554,0.2663,0.3913
7,SxA,3,7.0132,0.0715,0.1603
8,SxH,3,6.9102,0.0748,0.1603
9,AxH,1,0.2569,0.6122,0.7064


True

### Between-run check
Conditional on each condition's total, the successes in the first run are hypergeometric.

In [4]:
pairs = np.array(cond.pair.tolist(), dtype=float)
phat = y / n; interior = (y > 0) & (y < n)
Q = float(np.sum((pairs[interior, 0] - pairs[interior, 1]) ** 2
                 / (20 * phat[interior] * (1 - phat[interior]))))
rng = np.random.default_rng(20261004); B = 100000
sim = rng.hypergeometric(y[interior].astype(int), (20 - y[interior]).astype(int), 10,
                         size=(B, int(interior.sum())))
qsim = np.sum((2 * sim - y[interior]) ** 2 / (20 * phat[interior] * (1 - phat[interior])), axis=1)
p_Q = (1 + np.sum(qsim >= Q - 1e-10)) / (B + 1)
print(f"Q = {Q:.2f} over {interior.sum()} nonboundary conditions; conditional p = {p_Q:.3f}")
check("Replicate statistic Q", Q, 53.31, 0.005)
check("Nonboundary conditions", interior.sum(), 46, 0)
check("Conditional upper-tail p for Q", p_Q, 0.255, 0.0005)

Q = 53.31 over 46 nonboundary conditions; conditional p = 0.255


True

### How much interaction structure the observed rates contain

In [5]:
def firth_nll(b, Z):
    eta = Z @ b; p = expit(eta); w = n * p * (1 - p)
    sign, ld = np.linalg.slogdet(Z.T @ (w[:, None] * Z))
    return np.inf if sign <= 0 else float(np.sum(n * np.logaddexp(0, eta) - y * eta) - 0.5 * ld)
def firth_score(b, Z):
    p = expit(Z @ b); w = n * p * (1 - p); I = Z.T @ (w[:, None] * Z)
    h = w * np.sum(Z * np.linalg.solve(I, Z.T).T, axis=1)
    return -(Z.T @ (y - n * p + h * (0.5 - p)))
fitted = {}
for order in range(1, 5):
    Z = np.column_stack([np.ones(64), *[z for t, z in terms.items() if len(t.split(":")) <= order]])
    opt = minimize(firth_nll, np.zeros(Z.shape[1]), args=(Z,), jac=firth_score, method="BFGS",
                   options={"gtol": 1e-8, "maxiter": 3000})
    fitted[order] = expit(Z @ opt.x)
order_tab = pd.DataFrame([dict(order=o, max_abs_departure_pts=100 * np.max(np.abs(fitted[o] - y / n)))
                          for o in fitted])
display(order_tab.round(2))
for o, rep in zip(range(1, 5), [31.1, 16.3, 7.4, 2.4]):
    check(f"Largest departure, interactions up to order {o} (points)",
          order_tab.loc[o - 1, "max_abs_departure_pts"], rep, 0.05)
check("Largest change, adding four-factor term (points)", 100 * np.max(np.abs(fitted[4] - fitted[3])), 5.7, 0.05)

,order,max_abs_departure_pts
0,1,31.11
1,2,16.27
2,3,7.36
3,4,2.38


True

## 3. Condition-level results (Section 4.3; Table 6; Supplementary Table S5)

In [6]:
cond["rate_pct"] = 100 * y / n
cond["cp_lower_pct"] = 100 * np.where(y > 0, beta.ppf(0.025, y, n - y + 1), 0)
cond["cp_upper_pct"] = 100 * np.where(y < n, beta.ppf(0.975, y + 1, n - y), 1)
tableS5 = cond.drop(columns="pair").rename(columns={"s": "Successes_of_20"})
tableS5.to_csv(RES / "TableS5_condition_outcomes.csv", index=False)
complete = tableS5[tableS5.Successes_of_20 == 20]
complete.to_csv(RES / "Table6_complete_success_conditions.csv", index=False)
display(complete[KEYS])
check("Conditions with 20/20", len(complete), 18, 0)
check("Royal-blue trapezium successes (of 80)",
      runs[(runs.Color == "Royal Blue") & (runs.Config == "Trapezium")].Success.sum(), 80, 0)
check("Exact 95% lower bound at 20/20 (%)", complete.cp_lower_pct.iloc[0], 83.2, 0.05)

,Color,Config,Ambient,Height
10,Lime Green,Rectangular,360,250
12,Lime Green,Trapezium,110,250
13,Lime Green,Trapezium,110,300
14,Lime Green,Trapezium,360,250
24,Pink,Rectangular,110,250
26,Pink,Rectangular,360,250
28,Pink,Trapezium,110,250
29,Pink,Trapezium,110,300
33,Red,Cylindrical,110,300
36,Red,Hexagonal,110,250


True

## 4. Run-order check (Supplementary Section S2)
The run order was randomized in Excel within the two fixture groups. A logistic trend on
standardized position is fitted with one indicator per condition, so it compares runs
within conditions.

In [7]:
d = runs.sort_values("Sl").reset_index(drop=True)
grp = d.Config.isin(["Rectangular", "Cylindrical"])
assert grp[:64].all() and not grp[64:].any()  # rectangular/cylindrical first, then trapezium/hexagonal
pos = ((d.Sl - d.Sl.mean()) / d.Sl.std()).to_numpy()[:, None]
cid = pd.get_dummies(d[KEYS].astype(str).agg("|".join, axis=1)).to_numpy(float)
endog = np.column_stack([d.Success, 10 - d.Success])
f0 = sm.GLM(endog, cid, family=sm.families.Binomial()).fit(maxiter=300)
f1 = sm.GLM(endog, np.column_stack([cid, pos]), family=sm.families.Binomial()).fit(maxiter=300)
LR = 2 * (f1.llf - f0.llf)
print(f"slope {f1.params[-1]:+.3f} per SD, LR chi2(1) = {LR:.3f}, p = {chi2.sf(LR, 1):.3f}")
check("Run-order slope (per SD of position)", f1.params[-1], -0.21, 0.005)
check("Run-order likelihood-ratio statistic", LR, 0.85, 0.005)
check("Run-order p", chi2.sf(LR, 1), 0.36, 0.005)
check("Successes in runs 1-64 (of 640)", d.Success[:64].sum(), 489, 0)
check("Successes in runs 65-128 (of 640)", d.Success[64:].sum(), 565, 0)

slope -0.210 per SD, LR chi2(1) = 0.849, p = 0.357


True

## 5. Transformed-count ANOVA (Supplementary Section S3; Tables S6–S8)

In [8]:
a = runs.rename(columns={"Config": "Configuration", "Ambient": "Illuminance"}).copy()
a["z"] = (a.Success.astype(float) ** 3 - 1) / 3
F4 = ["Color", "Configuration", "Illuminance", "Height"]
full = ols("z ~ C(Color, Sum)*C(Configuration, Sum)*C(Illuminance, Sum)*C(Height, Sum)", a).fit()
reduced = ols("z ~ C(Color, Sum)*C(Configuration, Sum) + C(Illuminance, Sum) + C(Height, Sum)"
              " + C(Color, Sum):C(Height, Sum) + C(Configuration, Sum):C(Illuminance, Sum)"
              " + C(Configuration, Sum):C(Height, Sum)", a).fit()
short = {"C(Color, Sum)": "C", "C(Configuration, Sum)": "S", "C(Illuminance, Sum)": "A",
         "C(Height, Sum)": "H", "Residual": "Pure error"}
def tidy(t):
    t = t.drop("Intercept")
    t.index = [":".join(short.get(x, x) for x in i.split(":")).replace(":", "x") for i in t.index]
    t["mean_sq"] = t.sum_sq / t.df
    return t
af = tidy(anova_lm(full, typ=3))
ar = tidy(anova_lm(reduced, typ=3)).rename(index={"Pure error": "Error"})
eff = af.drop(index="Pure error").copy()
eff["p_BH"] = multipletests(eff["PR(>F)"], method="fdr_bh")[1]
af.to_csv(RES / "TableS6_full_ANOVA.csv"); ar.to_csv(RES / "TableS7_reduced_ANOVA.csv")
display(af.round(3)); display(eff[["PR(>F)", "p_BH"]].round(4))
REPORTED_S6 = {"C": (152144, 14.37), "S": (545404, 51.52), "A": (30918, 8.76), "H": (99867, 28.30),
               "CxS": (181077, 5.70), "CxA": (26537, 2.51), "CxH": (40995, 3.87), "SxA": (64019, 6.05),
               "SxH": (70786, 6.69), "AxH": (87, 0.02), "CxSxA": (62998, 1.98), "CxSxH": (75441, 2.38),
               "CxAxH": (4385, 0.41), "SxAxH": (30553, 2.89), "CxSxAxH": (58110, 1.83)}
for term, (ss, F) in REPORTED_S6.items():
    check(f"Table S6 SS, {term}", af.loc[term, "sum_sq"], ss, 0.5)
    check(f"Table S6 F, {term}", af.loc[term, "F"], F, 0.005)
check("Pure-error SS", af.loc["Pure error", "sum_sq"], 225853.89, 0.005)
check("Pure-error mean square", af.loc["Pure error", "mean_sq"], 3528.97, 0.005)
check("Full-model R2 (%)", 100 * full.rsquared, 86.47, 0.005)
check("Full-model adjusted R2 (%)", 100 * full.rsquared_adj, 73.15, 0.005)
check("ANOVA p_BH, CxSxH", eff.loc["CxSxH", "p_BH"], 0.036, 0.0005)
check("ANOVA p_BH, SxAxH", eff.loc["SxAxH", "p_BH"], 0.064, 0.0005)
check("Zero residuals in full model", int(np.sum(np.abs(full.resid) < 1e-8)), 50, 0)
lof_ss = reduced.ssr - full.ssr; lof_F = (lof_ss / 37) / full.mse_resid
check("Reduced-model R2 (%)", 100 * reduced.rsquared, 71.01, 0.005)
check("Reduced-model adjusted R2 (%)", 100 * reduced.rsquared_adj, 63.54, 0.005)
check("Lack-of-fit SS", lof_ss, 258111.20, 0.005)
check("Lack-of-fit F(37,64)", lof_F, 1.98, 0.005)
check("Lack-of-fit p", fdist.sf(lof_F, 37, 64), 0.008, 0.0005)
check("Reduced-model error SS", ar.loc["Error", "sum_sq"], 483965, 0.5)

# Tukey marginal means on the transformed scale with full-model pure error
tuk = []
for factor in F4:
    g = a.groupby(factor).z.agg(["mean", "size"]); k = len(g)
    for l1, l2 in itertools.combinations(g.index, 2):
        se = np.sqrt(full.mse_resid / 2 * (1 / g.loc[l1, "size"] + 1 / g.loc[l2, "size"]))
        q = abs(g.loc[l1, "mean"] - g.loc[l2, "mean"]) / se
        tuk.append(dict(factor=factor, level_1=l1, level_2=l2, q=q, p_tukey=studentized_range.sf(q, k, 64)))
tuk = pd.DataFrame(tuk); tuk.to_csv(RES / "TableS8_Tukey_pairs.csv", index=False)
means = {f_: a.groupby(f_).z.mean() for f_ in F4}
for (f_, lev), rep in {("Color", "Red"): 281.05, ("Color", "Royal Blue"): 220.66, ("Color", "Pink"): 196.84,
                       ("Color", "Lime Green"): 196.59, ("Configuration", "Trapezium"): 313.80,
                       ("Configuration", "Rectangular"): 253.73, ("Configuration", "Hexagonal"): 184.42,
                       ("Configuration", "Cylindrical"): 143.20, ("Illuminance", 110): 239.33,
                       ("Illuminance", 360): 208.24, ("Height", 250): 251.72, ("Height", 300): 195.85}.items():
    check(f"Table S8 mean, {f_}={lev}", means[f_][lev], rep, 0.005)
hc = tuk[(tuk.factor == "Configuration") & tuk.level_1.isin(["Cylindrical", "Hexagonal"])
         & tuk.level_2.isin(["Cylindrical", "Hexagonal"])].p_tukey.iloc[0]
check("Tukey p, hexagonal vs cylindrical", hc, 0.035, 0.0005)
display(tuk.round(4))

,sum_sq,df,F,PR(>F),mean_sq
C,152144.094,3.0,14.371,0.000,50714.698
S,545403.906,3.0,51.517,0.000,181801.302
A,30917.556,1.0,8.761,0.004,30917.556
H,99867.253,1.0,28.299,0.000,99867.253
CxS,181077.191,9.0,5.701,0.000,20119.688
CxA,26537.375,3.0,2.507,0.067,8845.792
SxA,64018.646,3.0,6.047,0.001,21339.549
CxH,40995.455,3.0,3.872,0.013,13665.152
SxH,70786.309,3.0,6.686,0.001,23595.436
AxH,86.681,1.0,0.025,0.876,86.681


,PR(>F),p_BH
C,0.0000,0.0000
S,0.0000,0.0000
A,0.0043,0.0092
H,0.0000,0.0000
CxS,0.0000,0.0000
CxA,0.0668,0.0835
SxA,0.0011,0.0027
CxH,0.0131,0.0246
SxH,0.0005,0.0016
AxH,0.8760,0.8760


,factor,level_1,level_2,q,p_tukey
0,Color,Lime Green,Pink,0.0238,1.0000
1,Color,Lime Green,Red,8.0425,0.0000
2,Color,Lime Green,Royal Blue,2.2914,0.3747
3,Color,Pink,Red,8.0187,0.0000
4,Color,Pink,Royal Blue,2.2675,0.3840
5,Color,Red,Royal Blue,5.7512,0.0008
6,Configuration,Cylindrical,Hexagonal,3.9251,0.0355
7,Configuration,Cylindrical,Rectangular,10.5253,0.0000
8,Configuration,Cylindrical,Trapezium,16.2458,0.0000
9,Configuration,Hexagonal,Rectangular,6.6003,0.0001


## 6. Conditional prediction checks (Supplementary Section S4; Table S9)
Reduced structure (four main effects and four two-factor interactions), five-fold
cross-validation with both runs of a condition in the same fold (seed 42).

In [9]:
def onehot(values, levels): return np.array([[float(v == l) for l in levels] for v in values])
def design(frame):
    c = onehot(frame.Color, COLORS); s = onehot(frame.Config, CONFIGS)
    a_ = 2 * (frame.Ambient.to_numpy() == 360).astype(float) - 1
    h_ = 2 * (frame.Height.to_numpy() == 300).astype(float) - 1
    cols = [np.ones(len(frame))] + [c[:, i] for i in range(1, 4)] + [s[:, j] for j in range(1, 4)] + [a_, h_]
    cols += [c[:, i] * s[:, j] for i in range(1, 4) for j in range(1, 4)]
    cols += [c[:, i] * h_ for i in range(1, 4)] + [s[:, j] * a_ for j in range(1, 4)] + [s[:, j] * h_ for j in range(1, 4)]
    return np.column_stack(cols)
cv = runs.copy()
Xr = design(cv); yr = cv.Success.to_numpy(float)
groups = cv.groupby(KEYS).ngroup().to_numpy()
def folds(k=5, seed=42):
    gids = np.unique(groups); np.random.default_rng(seed).shuffle(gids)
    for fold in np.array_split(gids, k):
        test = np.isin(groups, fold); yield ~test, test
def irls(x, k, m, iters=50, ridge=1e-4):
    b = np.zeros(x.shape[1])
    for _ in range(iters):
        eta = x @ b; p = np.clip(expit(eta), 1e-6, 1 - 1e-6); w = m * p * (1 - p)
        zz = eta + (k - m * p) / np.clip(w, 1e-6, None)
        bn = np.linalg.solve(x.T @ (x * w[:, None]) + ridge * np.eye(x.shape[1]), x.T @ (w * zz))
        done = np.max(np.abs(bn - b)) < 1e-8; b = bn
        if done: break
    return b
pred_lin = np.zeros(128); pred_bin = np.zeros(128)
for tr, te in folds():
    pred_lin[te] = np.clip(Xr[te] @ np.linalg.solve(Xr[tr].T @ Xr[tr], Xr[tr].T @ yr[tr]), 0, 10)
    pred_bin[te] = 10 * expit(Xr[te] @ irls(Xr[tr], yr[tr], np.full(tr.sum(), 10.0)))
def metrics(p):
    e = yr - p
    return dict(MAE=np.mean(np.abs(e)), RMSE=np.sqrt(np.mean(e ** 2)), R2=1 - np.sum(e ** 2) / np.sum((yr - yr.mean()) ** 2))
tableS9 = pd.DataFrame([dict(Model="Stabilized binomial logistic", **metrics(pred_bin)),
                        dict(Model="Linear interaction model", **metrics(pred_lin))])
tableS9.to_csv(RES / "TableS9_cross_validation.csv", index=False)
display(tableS9.round(3))
for model, (mae, rmse, r2) in {"Stabilized binomial logistic": (1.185, 1.719, 0.454),
                               "Linear interaction model": (1.220, 1.695, 0.470)}.items():
    r = tableS9.set_index("Model").loc[model]
    check(f"Table S9 MAE, {model}", r.MAE, mae, 0.0005)
    check(f"Table S9 RMSE, {model}", r.RMSE, rmse, 0.0005)
    check(f"Table S9 R2, {model}", r.R2, r2, 0.0005)
cells = runs.groupby(KEYS).size().reset_index()[KEYS]
Xc = design(cells)
lin_full = Xc @ np.linalg.solve(Xr.T @ Xr, Xr.T @ yr)
zr = (yr ** 3 - 1) / 3
tr_full = Xc @ np.linalg.solve(Xr.T @ Xr, Xr.T @ zr)
check("Full-data linear predictions clipped to 10", int(np.sum(lin_full > 10)), 8, 0)
check("Transformed-model predictions reaching 10 after back-transformation",
      int(np.sum(np.clip((3 * np.clip(tr_full, -0.3, None) + 1) ** (1 / 3), 0, 10) >= 10)), 4, 0)
check("Transformed fit at royal-blue trapezium, 110 lx, 250 mm",
      tr_full[((cells.Color == "Royal Blue") & (cells.Config == "Trapezium") & (cells.Ambient == 110)
               & (cells.Height == 250)).to_numpy()][0], 369.45, 0.005)

,Model,MAE,RMSE,R2
0,Stabilized binomial logistic,1.185,1.719,0.454
1,Linear interaction model,1.220,1.695,0.470


True

## 7. Exploratory optical comparison (Supplementary Section S5; Table S10)

In [10]:
opt = pd.read_csv(DATA / "instrument_specific_power_ratios.csv").set_index("Color")
cols = ["ratio_450_setting_pct", "ratio_550_setting_pct", "ratio_650_setting_pct"]
for wl, c_ in zip(["450", "550", "650"], cols):
    assert np.allclose(opt[c_], 100 * opt[f"power_{wl}_uW"] / opt[f"reference_{wl}_uW"], atol=1e-5)
opt["mean_ratio_pct"] = opt[cols].mean(axis=1)
opt["spread_pts"] = opt[cols].max(axis=1) - opt[cols].min(axis=1)
opt.to_csv(RES / "TableS10_optical_ratios.csv")
display(opt[cols + ["mean_ratio_pct", "spread_pts", "assembly_rate_pct"]].round(3))
for color, rep in {"Pink": 7.4, "Lime Green": 4.9, "Red": 3.4, "Royal Blue": 1.5}.items():
    check(f"Table S10 mean ratio, {color} (%)", opt.loc[color, "mean_ratio_pct"], rep, 0.05)
check("Largest spread across meter settings (points)", opt.spread_pts.max(), 0.26, 0.005)

,ratio_450_setting_pct,ratio_550_setting_pct,ratio_650_setting_pct,mean_ratio_pct,spread_pts,assembly_rate_pct
Color,,,,,,
Lime Green,5.089,4.834,4.922,4.948,0.255,76.562
Pink,7.429,7.306,7.374,7.370,0.122,76.562
Red,3.393,3.432,3.430,3.418,0.039,94.062
Royal Blue,1.432,1.469,1.480,1.460,0.048,82.188


True

## Summary of checks

In [11]:
summary = pd.DataFrame(CHECKS)
summary.to_csv(RES / "reproduction_checks.csv", index=False)
print(f"{summary.ok.sum()} of {len(summary)} reported values reproduced.")
if not summary.ok.all():
    display(summary[~summary.ok])
assert summary.ok.all(), "Some reported values were not reproduced; see the table above."

160 of 160 reported values reproduced.
